# MegaDetector V6 para la app Android (TFLite)

Este notebook convierte **MegaDetector V6 (YOLOv9-c)** —el mismo detector que usamos para limpiar el dataset— a **TensorFlow Lite**, para usarlo dentro de la app.

En la app servirá como "portero" antes del clasificador:
- **No hay animal** → *"No se detecta ningún animal, toma otra foto"*.
- **Persona** → no se clasifica.
- **Animal** → se recorta (10 % de margen, igual que el dataset) y pasa al clasificador de 40 especies.

**Antes de ejecutar:** en el panel derecho → *Settings* → **Internet: On**. No hace falta GPU.
Luego: **Run All**. Al final, en la pestaña **Output**, descarguen `megadetector.tflite` y `megadetector_info.json`.

In [ ]:
# 0. Revisar que el notebook tenga internet
import socket
try:
    socket.create_connection(("pypi.org", 443), timeout=10).close()
    print("Internet: OK")
except OSError:
    raise SystemExit("SIN INTERNET: en el panel derecho -> Settings -> Internet: On. "
                     "Si ya está activado, menú Run -> Restart & clear cell outputs y vuelvan a hacer Run All.")

In [ ]:
# 1. Instalar ultralytics y buscar las direcciones de los pesos de MegaDetector V6
import subprocess, sys, re, pathlib, importlib.util
from urllib.parse import urlparse
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "PytorchWildlife"], check=True)

VERSION = "yolov9"   # si la exportación falla, cambien a "yolov10" y vuelvan a ejecutar todo

raiz = pathlib.Path(importlib.util.find_spec("PytorchWildlife").origin).parent
urls = set()
for f in raiz.rglob("*.py"):
    for u in re.findall(r'https?://[^\s"\'\)<>{}]+', f.read_text(errors="ignore")):
        if "mdv6" in u.lower() or "megadetectorv6" in u.lower():
            urls.add(u.rstrip(".,;"))
print("Direcciones encontradas:")
print(*sorted(urls), sep="\n")
candidatas = [u for u in sorted(urls) if re.search(VERSION + r"[-_]?c", u.lower()) and "." in (urlparse(u).hostname or "")]
# Solo la versión de ultralytics (.pt). Las versiones "mit" / "apa" (.ckpt) usan otra librería que no sirve aquí.
def es_ultralytics(u):
    n = urlparse(u).path.lower()
    return n.split("?")[0].endswith(".pt") and "-mit-" not in n and "-apa-" not in n
print("\nDescartadas (otra librería):", [u for u in candidatas if not es_ultralytics(u)])
candidatas = [u for u in candidatas if es_ultralytics(u)]
print("\nCandidatas:", candidatas or "ninguna (se usará el método alternativo)")

In [ ]:
# 2. Descargar los pesos (.pt): primero por dirección directa; si falla, con PytorchWildlife
import glob, os, torch
PT = "/kaggle/working/mdv6.pt"
descargado = False
for u in candidatas:
    try:
        print("Probando:", u)
        torch.hub.download_url_to_file(u, PT)
        descargado = True
        break
    except Exception as e:
        print("  no funcionó:", e)

if not descargado:
    print("\nMétodo alternativo: PytorchWildlife descarga el modelo por su cuenta…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PytorchWildlife"], check=True)
    from PytorchWildlife.models import detection as pw_detection
    pw_detection.MegaDetectorV6(device="cpu", pretrained=True, version=f"MDV6-{VERSION}-c")
    archivos = [p for p in glob.glob(os.path.expanduser("~/**/*.pt"), recursive=True) + glob.glob("/kaggle/**/*.pt", recursive=True)
                + glob.glob("/root/**/*.pt", recursive=True) + glob.glob("/tmp/**/*.pt", recursive=True)
                if "mdv6" in p.lower() or "megadetector" in p.lower()]
    print("Archivos encontrados:", archivos)
    PT = archivos[0]
print("Pesos:", PT, round(os.path.getsize(PT) / 1e6, 1), "MB")

In [ ]:
# 3 y 4. Abrir el modelo con ultralytics 8.3.100 y exportarlo a TFLite (en un proceso aparte).
import json

SCRIPT = "/kaggle/working/exportar_md.py"
with open(SCRIPT, "w") as f:
    f.write("\n".join([
        "import sys, types, importlib, json",
        "# El módulo 'imp' ya no existe en Python 3.13; una librería vieja lo pide al final de la exportación.",
        "if 'imp' not in sys.modules:",
        "    imp = types.ModuleType('imp'); imp.reload = importlib.reload; sys.modules['imp'] = imp",
        "import ultralytics",
        "from ultralytics import YOLO",
        "from ultralytics.utils import ASSETS",
        "m = YOLO(sys.argv[1])",
        "print('ULTRA', ultralytics.__version__, flush=True)",
        "print('NOMBRES', json.dumps({int(k): v for k, v in m.names.items()}), flush=True)",
        "r = m.predict(str(ASSETS / 'bus.jpg'), conf=0.2, verbose=False)[0]",
        "print('PRUEBA', [(m.names[int(c)], round(float(p), 2)) for c, p in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist())], flush=True)",
        "salida = m.export(format='tflite', imgsz=640, half=True)",
        "print('TFLITE', salida, flush=True)",
    ]))

def buscar_tflite():
    c = sorted(glob.glob("/kaggle/working/**/*float16.tflite", recursive=True))
    return c[0] if c else None

NOMBRES, TFL = None, None
for v in ["8.3.100", "8.3.40", "8.3.0"]:
    destino = f"/kaggle/working/ul_{v}"
    print(f"\n=== Probando ultralytics {v} ===")
    p = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", destino,
                        f"ultralytics=={v}", "ultralytics-thop"], capture_output=True, text=True)
    if p.returncode != 0:
        print("  no se pudo instalar:", p.stderr[-300:])
        continue
    env = dict(os.environ, PYTHONPATH=destino)
    p = subprocess.run([sys.executable, SCRIPT, PT], capture_output=True, text=True, env=env)
    for l in p.stdout.splitlines():
        if l.split(" ")[0] in ("ULTRA", "NOMBRES", "PRUEBA", "TFLITE"):
            print(" ", l)
        if l.startswith("NOMBRES "):
            NOMBRES = {int(k): x for k, x in json.loads(l[8:]).items()}
    TFL = buscar_tflite()
    if TFL:          # se detiene apenas existe el archivo, aunque haya fallado un paso posterior
        break
    print("  falló:", (p.stderr or p.stdout)[-800:])

assert TFL, "No se pudo exportar. Envíen pantallazo de esta celda."
print("\nArchivo TFLite:", TFL, round(os.path.getsize(TFL) / 1e6, 1), "MB")
print("Clases:", NOMBRES)


In [ ]:
# 5. Probar el TFLite igual que lo hará la app y guardar la información para la app
import json, shutil, numpy as np, tensorflow as tf
from PIL import Image
from ultralytics.utils import ASSETS

interp = tf.lite.Interpreter(model_path=TFL)
interp.allocate_tensors()
ent, sal = interp.get_input_details()[0], interp.get_output_details()
print("Entrada:", ent["shape"], ent["dtype"])
for s in sal: print("Salida:", s["shape"], s["dtype"])

def letterbox(ruta, lado=640):
    im = Image.open(ruta).convert("RGB")
    e = lado / max(im.size)
    nw, nh = round(im.width * e), round(im.height * e)
    lienzo = Image.new("RGB", (lado, lado), (114, 114, 114))
    lienzo.paste(im.resize((nw, nh)), ((lado - nw) // 2, (lado - nh) // 2))
    return np.asarray(lienzo, dtype=np.float32)[None] / 255.0

def detectar(ruta):
    interp.set_tensor(ent["index"], letterbox(ruta))
    interp.invoke()
    o = interp.get_tensor(sal[0]["index"])[0]
    if o.shape[0] > o.shape[1]: o = o.T          # deja la forma (4 + clases, N)
    cajas, puntajes = o[:4], o[4:]
    print("  coordenadas máx:", round(float(cajas.max()), 3), "(≤ 1.5 significa normalizadas)")
    for k, nombre in NOMBRES.items():
        print(f"  {nombre:8s} confianza máx {puntajes[k].max():.2f}  ·  cajas > 0.40: {(puntajes[k] > 0.4).sum()}")
    return float(cajas.max())

print("\nbus.jpg (debe salir person y vehicle, no animal):")
coord_max = detectar(ASSETS / "bus.jpg")

# Si agregaron el dataset mamiferos-colombia-40 (Add Input), se prueba con una foto de animal.
fotos = glob.glob("/kaggle/input/**/*.jpg", recursive=True)[:1]
if fotos:
    print("\nFoto de animal del dataset:", fotos[0])
    detectar(fotos[0])

shutil.copy(TFL, "/kaggle/working/megadetector.tflite")
info = {
    "modelo": f"MegaDetector V6 ({VERSION}-c)",
    "clases": NOMBRES,
    "entrada": {"forma": [int(x) for x in ent["shape"]], "tipo": str(ent["dtype"].__name__), "valores": "RGB 0..1, letterbox gris 114"},
    "salida": [{"forma": [int(x) for x in s["shape"]], "tipo": str(s["dtype"].__name__)} for s in sal],
    "coordenadas_normalizadas": coord_max <= 1.5,
    "tamano_MB": round(os.path.getsize(TFL) / 1e6, 1),
}
json.dump(info, open("/kaggle/working/megadetector_info.json", "w"), indent=2, ensure_ascii=False)
print("\n", json.dumps(info, indent=2, ensure_ascii=False))
print("\nListo. Descarguen megadetector.tflite y megadetector_info.json desde la pestaña Output.")